# Movie Recommender Full-Stack Workflow Notebook

This notebook is the model-side source of truth for a stronger architecture:
- Model training in Python `.ipynb`
- JavaScript backend (Express) calling Python inference
- Vite + React frontend calling backend APIs

## 1) Set Up the Python Notebook Environment
## 2) Train and Validate the Model in `.ipynb`
## 3) Export the Model Artifacts for Inference
## 4) Create a Python Inference Entry Script
## 5) Scaffold the JavaScript Backend (Express)
## 6) Call the Python Inference Script from Node.js
## 7) Add API Routes, Validation, and Error Handling
## 8) Configure CORS and Environment Variables
## 9) Scaffold the Vite + React Frontend
## 10) Connect React UI to Backend API
## 11) Render Predictions and Request States
## 12) Run and Verify End-to-End Flow in VS Code

In [ ]:
# 1) Set Up the Python Notebook Environment
# If running for the first time in this kernel, uncomment installs:
# %pip install pandas numpy scikit-learn nltk

import os
import re
import pickle
import unicodedata

import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

from combine_datasets import build_combined_dataset

import nltk
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

BASE_DIR = os.getcwd()
MODEL_DIR = os.path.join(BASE_DIR, "model")
os.makedirs(MODEL_DIR, exist_ok=True)

print("Working directory:", BASE_DIR)
print("Model output directory:", MODEL_DIR)

In [ ]:
# 2) Train and Validate the Model in `.ipynb`
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))
punct_re = re.compile(r"[^a-zA-Z0-9\s]")

LANG_CODE_MAP = {
    "en": "English", "hi": "Hindi", "te": "Telugu", "ta": "Tamil",
    "ml": "Malayalam", "kn": "Kannada", "bn": "Bengali", "mr": "Marathi",
    "gu": "Gujarati", "pa": "Punjabi", "ur": "Urdu", "fr": "French",
    "de": "German", "es": "Spanish", "it": "Italian", "pt": "Portuguese",
    "ja": "Japanese", "ko": "Korean", "zh": "Chinese", "ru": "Russian",
}

def clean_and_stem(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = punct_re.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words and len(w) > 1]
    return " ".join(tokens)

df = build_combined_dataset(save=True)

for col in ["title", "display_title", "genres", "overview", "keywords", "original_language", "media_type"]:
    if col not in df:
        df[col] = ""

df["title"] = df["title"].fillna("").astype(str).str.strip()
df = df[df["title"] != ""].copy()
df["display_title"] = df["display_title"].fillna(df["title"]).astype(str)
df["genres"] = df["genres"].fillna("").astype(str)
df["overview"] = df["overview"].fillna("").astype(str)
df["keywords"] = df["keywords"].fillna("").astype(str)
df["original_language"] = df["original_language"].fillna("").astype(str)
df["media_type"] = df["media_type"].replace("", "movie").fillna("movie").astype(str).str.lower()

df["language_name"] = df["original_language"].str.lower().map(LANG_CODE_MAP).fillna(
    df["original_language"].replace("", "Unknown").str.title()
)

tags_input = (
    df["display_title"] + " " +
    df["genres"].str.replace("|", " ", regex=False) + " " +
    df["overview"] + " " +
    df["keywords"].str.replace("_", " ", regex=False) + " " +
    df["language_name"] + " " + df["language_name"] + " " +
    df["media_type"]
)

df["tags"] = tags_input.apply(clean_and_stem)
df = df[df["tags"].str.len() > 0].copy()

vectorizer = TfidfVectorizer(max_features=15000, stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["tags"])

knn_model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=30)
knn_model.fit(tfidf_matrix)

print("Training rows:", len(df))
print("TF-IDF shape:", tfidf_matrix.shape)

In [ ]:
# 3) Export the Model Artifacts for Inference
movies_for_inference = df[[
    "title", "display_title", "genres", "overview", "keywords", "language_name",
    "media_type", "vote_average", "popularity", "poster_url", "release_date", "year", "tags"
]].copy()

with open(os.path.join(MODEL_DIR, "movies.pkl"), "wb") as f:
    pickle.dump(movies_for_inference, f)

with open(os.path.join(MODEL_DIR, "knn_model.pkl"), "wb") as f:
    pickle.dump(knn_model, f)

with open(os.path.join(MODEL_DIR, "tfidf_matrix.pkl"), "wb") as f:
    pickle.dump(tfidf_matrix, f)

print("Saved: movies.pkl, knn_model.pkl, tfidf_matrix.pkl")

In [ ]:
# 4) Create a Python Inference Entry Script
# This cell writes a lightweight inference script if missing.
inference_script = os.path.join(BASE_DIR, "api_bridge.py")
print("Inference bridge path:", inference_script)
print("Exists:", os.path.exists(inference_script))

# 2) quick sanity validation from recommender import
from recommender import get_recommendations
sample = get_recommendations("Interstellar", top_n=5)
print("Sample recommendation count:", len(sample))
if sample:
    print("Top result:", sample[0].get("display_title"))

## 5) Scaffold the JavaScript Backend (Express)
Backend folder in this workspace: `backend/`
- `backend/src/server.js`: Express bootstrap + routes
- `backend/src/pythonService.js`: Calls `api_bridge.py` via `child_process`
- `backend/src/watchHistory.js`: Shared watch-history file adapter

## 6) Call the Python Inference Script from Node.js
Node executes:
- `python api_bridge.py recommend --query "Interstellar" --top-n 18`
- Expects JSON only on stdout.

## 7) Add API Routes, Validation, and Error Handling
Implemented routes:
- `GET /api/popular`
- `GET /api/personalized`
- `POST /api/recommend`
- `GET /api/autocomplete`
- `GET /api/watched`
- `POST /api/watch`
- `POST /api/unwatch`

## 8) Configure CORS and Environment Variables
Backend enables CORS and supports:
- `PORT` for backend port
- `PYTHON_EXECUTABLE` for custom python path

## 9) Scaffold the Vite + React Frontend
Frontend folder in this workspace: `frontend/`
- Vite + React setup
- Search, suggestions, recommendations, watched toggles

## 10) Connect React UI to Backend API
Vite proxy in `frontend/vite.config.js` forwards `/api` -> `http://localhost:4000`.

## 11) Render Predictions and Request States
React UI handles:
- Loading
- Success results
- Error messages
- Empty personalized state

## 12) Run and Verify End-to-End Flow in VS Code
Terminal 1:
```bash
cd backend
npm install
npm run dev
```
Terminal 2:
```bash
cd frontend
npm install
npm run dev
```
Open `http://localhost:5173` and search for a movie/filter query.